# Local-DDx v10 Benchmark on Open-XDDx

Runs the full 7-round diagnostic pipeline against the Open-XDDx dataset (570 cases) using vLLM on A100 GPU.

**Model:** `MaziyarPanahi/Meta-Llama-3-8B-Instruct-GPTQ` (same as v7.4 benchmarks)

**Steps:**
1. Install dependencies
2. Clone repo & verify GPU
3. Smoke test vLLM backend
4. Run pilot (10 cases)
5. Evaluate results
6. Compare with v7.4 baseline

In [ ]:
# Cell 1: Install dependencies
!pip install -q vllm pandas openpyxl pyyaml
print("Dependencies installed.")

In [ ]:
# Cell 2: Clone repo and verify GPU
import os
import torch

# Clone if not already present
if not os.path.exists('Local-DDx'):
    !git clone https://github.com/Baglecake/Local-DDx.git
else:
    print("Repo already cloned")

os.chdir('Local-DDx')
print(f"Working directory: {os.getcwd()}")

# Verify GPU
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU Memory: {mem_gb:.1f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime > Change runtime type > A100")

In [ ]:
# Cell 3: Verify files exist
import os

required_files = [
    'benchmark/benchmark_config.yaml',
    'benchmark/vllm_backend.py',
    'benchmark/batch_runner.py',
    'benchmark/ddx_evaluator.py',
    'v10_full_pipeline/data/Open-XDDx.xlsx',
    'v10_full_pipeline/Modules/ddx_core.py',
    'v10_full_pipeline/Modules/ddx_runner.py',
    'v9_ollama_ui/Modules/inference_backends.py',
]

all_ok = True
for f in required_files:
    exists = os.path.exists(f)
    status = 'OK' if exists else 'MISSING'
    print(f"  [{status}] {f}")
    if not exists:
        all_ok = False

# Check symlink (v10 -> v9 inference_backends)
v10_ib = 'v10_full_pipeline/Modules/inference_backends.py'
if os.path.islink(v10_ib):
    print(f"\n  Symlink: {v10_ib} -> {os.readlink(v10_ib)}")
elif os.path.exists(v10_ib):
    print(f"\n  {v10_ib} exists (not symlink, OK)")
else:
    print(f"\n  WARNING: {v10_ib} missing — creating copy")
    import shutil
    shutil.copy('v9_ollama_ui/Modules/inference_backends.py', v10_ib)

print(f"\nAll files present: {all_ok}")

In [ ]:
# Cell 4: Smoke test — load vLLM backend, generate one response
import sys
sys.path.insert(0, 'v9_ollama_ui/Modules')
sys.path.insert(0, 'v10_full_pipeline/Modules')
sys.path.insert(0, 'benchmark')

from vllm_backend import VLLMBackend
from inference_backends import SamplingConfig

print("Loading vLLM backend...")
backend = VLLMBackend(
    model_name='MaziyarPanahi/Meta-Llama-3-8B-Instruct-GPTQ',
    gpu_memory_utilization=0.90,
    max_model_len=4096,
    quantization='gptq',
    dtype='float16',
)

print(f"Backend available: {backend.is_available()}")
backend.load('MaziyarPanahi/Meta-Llama-3-8B-Instruct-GPTQ')

# Quick test
config = SamplingConfig(temperature=0.7, max_tokens=100)
messages = [
    {'role': 'system', 'content': 'You are a medical specialist. Be concise.'},
    {'role': 'user', 'content': 'Name three common symptoms of pneumonia.'}
]
response = backend.generate_chat(messages, config)
print(f"\nSmoke test response:\n{response[:300]}")
print(f"\nBackend info: {backend.get_info()}")

In [ ]:
# Cell 5: Run pilot benchmark (10 cases)
# Uses batch_runner.py with checkpoint/resume support

!python benchmark/batch_runner.py \
    --config benchmark/benchmark_config.yaml \
    --cases 0-9

In [ ]:
# Cell 6: Evaluate pilot results

!python benchmark/ddx_evaluator.py \
    --results-dir benchmark/results \
    --dataset v10_full_pipeline/data/Open-XDDx.xlsx \
    --output benchmark/results/pilot_report.json

In [ ]:
# Cell 7: Display results as DataFrame + comparison chart
import json
import pandas as pd

report_path = 'benchmark/results/pilot_report.json'
with open(report_path) as f:
    report = json.load(f)

# Per-case results table
df = pd.DataFrame(report['per_case'])
display_cols = ['case_index', 'tp', 'fp', 'fn', 'ae',
                'recall', 'precision', 'f1', 'crq', 'safety', 'duration']
print("Per-Case Results:")
display(df[display_cols].round(3))

# Summary
s = report['summary']
print(f"\n--- Summary ---")
print(f"Macro Recall:    {s['macro_recall']:.1%}")
print(f"Macro Precision: {s['macro_precision']:.1%}")
print(f"Macro F1:        {s['macro_f1']:.1%}")
print(f"Macro CRQ:       {s['macro_crq']:.1%}")
print(f"Macro Safety:    {s['macro_safety']:.1%}")

# v7.4 comparison
v74 = report['v74_baseline']
print(f"\n--- v7.4 Comparison ---")
print(f"Recall:    v7.4={v74['recall']:.1%}  v10={s['macro_recall']:.1%}  "
      f"delta={s['macro_recall']-v74['recall']:+.1%}")
print(f"Precision: v7.4={v74['precision']:.1%}  v10={s['macro_precision']:.1%}  "
      f"delta={s['macro_precision']-v74['precision']:+.1%}")

## Full Run (570 cases)

Once the pilot looks good, run the full dataset. Use `--resume` if the session disconnects.

**Estimated time:** ~8-16 hours for 570 cases (depending on pipeline complexity)

**Recommended approach:** Run in chunks of 100 to minimize risk:
```
!python benchmark/batch_runner.py --config benchmark/benchmark_config.yaml --cases 0-99
!python benchmark/batch_runner.py --config benchmark/benchmark_config.yaml --cases 100-199 --resume
!python benchmark/batch_runner.py --config benchmark/benchmark_config.yaml --cases 200-299 --resume
...
```

In [ ]:
# Cell 8: Full run (uncomment when ready)

# !python benchmark/batch_runner.py \
#     --config benchmark/benchmark_config.yaml \
#     --cases 0-569 \
#     --resume

In [ ]:
# Cell 9: Evaluate full run

# !python benchmark/ddx_evaluator.py \
#     --results-dir benchmark/results \
#     --dataset v10_full_pipeline/data/Open-XDDx.xlsx \
#     --output benchmark/results/full_report.json

In [ ]:
# Cell 10: Download results (run this before session ends!)
import shutil
import os

results_dir = 'benchmark/results'
if os.path.exists(results_dir):
    archive = shutil.make_archive('benchmark_results', 'zip', results_dir)
    print(f"Results archived: {archive}")
    print("Download from the Files panel on the left, or:")
    try:
        from google.colab import files
        files.download(archive)
    except ImportError:
        print("  (Not in Colab — archive created locally)")
else:
    print("No results directory found")